# Bias Audit with Linear Regression
## EU AI Act Article 10 — Veri Yonetisimi ve Onyargi Kontrolu

**Proje:** Kredi riski tahmin modelindeki demografik onyargiyi lineer regresyonla tespit etmek
**Veri seti:** German Credit Dataset (UCI)
**ORIENT Asamasi:** Evaluate — "Mevcut sistemde uyum aciklari nerede?"
**Hazirlayan:** Hexis AI Governance (hexis.center)

---

### Bu Notebook'ta Ne Ogreneceksin?

1. Bir kredi risk modelinin nasil kuruldugunu
2. Korunan ozelliklerin (cinsiyet, yas) model kararlarini nasil etkiledigini
3. Lineer regresyon katsayilarinin "bias kaniti" olarak nasil okundugunu
4. EU AI Act'in bu konuda ne gerektirdigini

### Neden Bu Proje Onemli?

EU AI Act Article 10, yuksek riskli AI sistemlerinde kullanilan egitim verisinin
onyargi acisindan incelenmesini zorunlu kiliyor. Kredi skorlama sistemleri
Annex III kapsaminda **yuksek riskli** kategoride. Bu notebook, boyle bir
sistemde bias'in nasil tespit edilebilecegini gosteriyor.

---
## Bolum 1: Kurulum ve Veri Yukleme

German Credit Dataset, 1000 kredi basvurusunu icerir. Her basvuru icin
20 ozellik ve bir sonuc (iyi kredi / kotu kredi) bulunur.

Korunan ozellikler: **yas** ve **cinsiyet** — bunlar EU AI Act ve
esitlik mevzuati kapsaminda ayrimciliga yol acmamasi gereken degiskenlerdir.

In [ ]:
# Gerekli kutuphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Gorsellestirme ayarlari
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Kutuphaneler yuklendi')

In [ ]:
# German Credit Dataset'i yukle
# UCI sunucusu bazen kapali olabiliyor, birden fazla kaynak deniyoruz

columns = [
    'checking_account',    # Mevcut hesap durumu
    'duration_months',     # Kredi suresi (ay)
    'credit_history',      # Kredi gecmisi
    'purpose',             # Kredi amaci
    'credit_amount',       # Kredi miktari
    'savings_account',     # Tasarruf hesabi
    'employment_years',    # Istihdam suresi
    'installment_rate',    # Taksit orani (gelire gore %)
    'personal_status_sex', # Medeni hal + cinsiyet (KORUNAN OZELLIK)
    'other_debtors',       # Diger borclular/garantorler
    'residence_years',     # Ikamet suresi
    'property',            # Mulk durumu
    'age',                 # Yas (KORUNAN OZELLIK)
    'other_installments',  # Diger taksit planlari
    'housing',             # Konut durumu
    'existing_credits',    # Mevcut kredi sayisi
    'job',                 # Is turu
    'num_dependents',      # Bagimli kisi sayisi
    'telephone',           # Telefon
    'foreign_worker',      # Yabanci isci
    'credit_risk'          # HEDEF: 1=Iyi, 2=Kotu
]

# Birden fazla kaynak dene
urls = [
    'https://raw.githubusercontent.com/propublica/compas-analysis/master/german-credit-data/german.data',
    'https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv',
    'https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data'
]

df = None
for i, url in enumerate(urls):
    try:
        # german.csv (jbrownlee) virgul ile ayrilmis ve baslik satiri yok
        sep = ',' if url.endswith('.csv') else ' '
        df = pd.read_csv(url, sep=sep, header=None, names=columns)
        print(f'Veri seti yuklendi (kaynak {i+1})')
        break
    except Exception as e:
        print(f'Kaynak {i+1} basarisiz: {e}')

# Hicbiri calismadiysa sklearn uzerinden dene
if df is None:
    print('Online kaynaklar basarisiz. sklearn\'den South African Heart veri seti deneniyor...')
    from sklearn.datasets import fetch_openml
    data = fetch_openml('credit-g', version=1, as_frame=True)
    df = data.frame
    print('OpenML uzerinden yuklendi')

print(f'\nVeri seti boyutu: {df.shape[0]} satir, {df.shape[1]} sutun')
print(f'\nHedef degisken dagilimi:')
print(df['credit_risk'].value_counts())

---
## Bolum 2: Korunan Ozellikleri Anlamak

**Korunan ozellik (protected attribute):** Hukuki olarak ayrimcilik yapilmamasi
gereken demografik ozellikler. EU AI Act ve Turk hukukunda (Anayasa md. 10,
Is Kanunu md. 5) cinsiyet, yas, etnik koken gibi ozellikler korunur.

Bu veri setinde iki korunan ozellik var:
- **personal_status_sex:** Cinsiyet + medeni hal (birlesik kodlanmis)
- **age:** Yas (surekli degisken)

Simdi bunlari ayiklayalim ve dagilimlarini inceleyelim.

In [ ]:
# Cinsiyet bilgisini ayikla
# A91: erkek-bosanmis, A92: kadin-bosanmis/evli,
# A93: erkek-bekar, A94: erkek-evli, A95: kadin-bekar
df['gender'] = df['personal_status_sex'].map({
    'A91': 'male',
    'A92': 'female',
    'A93': 'male',
    'A94': 'male',
    'A95': 'female'
})

# Hedef degiskeni donustur: 1=Iyi -> 1, 2=Kotu -> 0
df['credit_good'] = (df['credit_risk'] == 1).astype(int)

# Yas gruplari olustur
df['age_group'] = pd.cut(df['age'], bins=[0, 25, 35, 45, 100],
                         labels=['25 alti', '25-35', '35-45', '45+'])

print('=== Cinsiyet Dagilimi ===')
print(df['gender'].value_counts())
print(f'\n=== Yas Istatistikleri ===')
print(df['age'].describe().round(1))

In [ ]:
# Ilk bias gostergesi: Kredi onay oranlari gruplara gore farkli mi?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cinsiyet bazinda onay orani
gender_approval = df.groupby('gender')['credit_good'].mean()
gender_approval.plot(kind='bar', ax=axes[0], color=['#686662', '#1C1E23'], edgecolor='#1C1E23')
axes[0].set_title('Kredi Onay Orani - Cinsiyete Gore', fontsize=13)
axes[0].set_ylabel('Onay Orani')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=0)
for i, v in enumerate(gender_approval):
    axes[0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Yas grubu bazinda onay orani
age_approval = df.groupby('age_group')['credit_good'].mean()
age_approval.plot(kind='bar', ax=axes[1], color='#686662', edgecolor='#1C1E23')
axes[1].set_title('Kredi Onay Orani - Yas Grubuna Gore', fontsize=13)
axes[1].set_ylabel('Onay Orani')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(age_approval):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.suptitle('EU AI Act Art. 10 - Veri Setinde Demografik Farklar',
             y=1.03, fontsize=14, fontweight='bold')
plt.show()

print('\nYorum: Eger gruplar arasinda belirgin fark varsa,')
print('bu henuz bias kaniti DEGIL - ama arastirilmasi gereken bir sinyal.')

---
## Bolum 3: Lineer Regresyon ile Bias Tespiti

### Temel Fikir

Lineer regresyonda her degiskenin bir **katsayisi (coefficient)** vardir.
Bu katsayi, "diger her sey sabitken, bu degisken sonucu ne kadar etkiliyor?"
sorusunu cevaplar.

Eger `cinsiyet` degiskeninin katsayisi istatistiksel olarak anlamli ve
buyukse, bu su anlama gelir: **Model, kredi kararinda cinsiyeti bir faktor
olarak kullaniyor** — bu da potansiyel bir bias gostergesi.

### Neden Lineer Regresyon?

- Katsayilar dogrudan yorumlanabilir ("kadin olmak kredi skorunu X kadar etkiliyor")
- Seffaf ve aciklanabilir — EU AI Act'in istedigi tam da bu
- Daha karmasik modellerde (random forest, neural network) bu kadar net yorum yapmak zor

In [ ]:
# Modelleme icin veri hazirligi

# Sayisal ozellikler
numerical_features = ['duration_months', 'credit_amount', 'installment_rate',
                       'residence_years', 'age', 'existing_credits', 'num_dependents']

# Cinsiyet: binary degisken (female=1, male=0)
df['is_female'] = (df['gender'] == 'female').astype(int)

# Yas: genc grubu (25 alti = 1, diger = 0)
df['is_young'] = (df['age'] < 25).astype(int)

# Feature matrix
feature_columns = numerical_features + ['is_female', 'is_young']
X = df[feature_columns].copy()
y = df['credit_good'].copy()

# Olceklendirme (katsayilari karsilastirabilir yapar)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_columns)

# Train/test ayirimi
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Egitim seti: {X_train.shape[0]} ornek')
print(f'Test seti: {X_test.shape[0]} ornek')
print(f'\nKullanilan ozellikler ({len(feature_columns)} adet):')
for f in feature_columns:
    marker = ' <-- KORUNAN' if f in ['is_female', 'is_young', 'age'] else ''
    print(f'  - {f}{marker}')

In [ ]:
# MODEL 1: Tum ozelliklerle lineer regresyon (bias dahil)
model_with_protected = LinearRegression()
model_with_protected.fit(X_train, y_train)

# Katsayilari incele
coef_df = pd.DataFrame({
    'Ozellik': feature_columns,
    'Katsayi': model_with_protected.coef_,
    'Mutlak Etki': np.abs(model_with_protected.coef_)
}).sort_values('Mutlak Etki', ascending=True)

# Katsayi grafigi
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#B2986C' if f in ['is_female', 'is_young', 'age'] else '#686662'
          for f in coef_df['Ozellik']]
bars = ax.barh(coef_df['Ozellik'], coef_df['Katsayi'], color=colors, edgecolor='#1C1E23')
ax.axvline(x=0, color='#1C1E23', linewidth=1)
ax.set_xlabel('Standartlastirilmis Katsayi')
ax.set_title('Lineer Regresyon Katsayilari - Kredi Onay Tahmini\n'
             '(Korunan ozellikler altin renkle isaretli)', fontsize=13)

# Degerleri goster
for bar, val in zip(bars, coef_df['Katsayi']):
    x_pos = val + 0.005 if val >= 0 else val - 0.005
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', ha=ha, fontsize=10)

plt.tight_layout()
plt.show()

print('\nKatsayi Yorumu:')
print('  Pozitif katsayi -> Kredi onay olasiligini ARTTIRIR')
print('  Negatif katsayi -> Kredi onay olasiligini AZALTIR')
print('  Sifira yakin -> Etkisi az')

# Korunan ozelliklerin etkisini vurgula
print('\nKorunan Ozelliklerin Etkisi:')
for _, row in coef_df[coef_df['Ozellik'].isin(['is_female', 'is_young', 'age'])].iterrows():
    direction = 'ARTTIRIR' if row['Katsayi'] > 0 else 'AZALTIR'
    print(f'  {row["Ozellik"]}: {row["Katsayi"]:.4f} -> Kredi onayini {direction}')

---
## Bolum 4: Bias Kanitini Olcmek

Katsayilari gorduk, ama istatistiksel olarak anlamli mi?
Bunun icin p-degerlerine bakmamiz gerekiyor.

Ayrica iki model karsilastirmasi yapacagiz:
- **Model A:** Korunan ozellikler DAHIL (potansiyel biased model)
- **Model B:** Korunan ozellikler HARIC (fairness-aware model)

Performans farki bize "bias'i kaldirmanin maliyeti"ni gosterir —
bu, gercek dunyada governance kararlarinin temelidir.

In [ ]:
# statsmodels ile detayli istatistiksel analiz
import statsmodels.api as sm

# Sabit terim ekle
X_train_sm = sm.add_constant(X_train)

# OLS regresyon
ols_model = sm.OLS(y_train, X_train_sm).fit()

# Sonuc tablosu
results_df = pd.DataFrame({
    'Katsayi': ols_model.params,
    'Std. Hata': ols_model.bse,
    't-degeri': ols_model.tvalues,
    'p-degeri': ols_model.pvalues,
    'Anlamli mi?': ['Evet' if p < 0.05 else 'Hayir' for p in ols_model.pvalues]
}).round(4)

print('=== OLS Regresyon Sonuclari ===')
print(f'R-kare skoru: {ols_model.rsquared:.4f}')
print(f'Duzeltilmis R-kare: {ols_model.rsquared_adj:.4f}')
print(f'\n--- Katsayi Tablosu (p < 0.05 anlamli) ---')
print(results_df.to_string())

# Korunan ozellikleri vurgula
print('\nBIAS ANALIZI - Korunan Ozellikler:')
for feat in ['is_female', 'is_young', 'age']:
    if feat in results_df.index:
        row = results_df.loc[feat]
        significance = 'ISTATISTIKSEL OLARAK ANLAMLI' if row['p-degeri'] < 0.05 else 'anlamli degil'
        print(f'\n  {feat}:')
        print(f'    Katsayi: {row["Katsayi"]:.4f} | p-degeri: {row["p-degeri"]:.4f}')
        print(f'    -> {significance}')
        if row['p-degeri'] < 0.05:
            print(f'    !! EU AI Act Art. 10 kapsaminda incelenmeli!')

In [ ]:
# MODEL KARSILASTIRMASI: Bias dahil vs. Bias haric

# Model A: Tum ozellikler (korunan dahil) - zaten yukarda egittik
protected_features = ['is_female', 'is_young', 'age']
y_pred_A = model_with_protected.predict(X_test)
y_pred_A_binary = (y_pred_A >= 0.5).astype(int)

# Model B: Korunan ozellikler haric
fair_features = [f for f in feature_columns if f not in protected_features]
model_fair = LinearRegression()
model_fair.fit(X_train[fair_features], y_train)
y_pred_B = model_fair.predict(X_test[fair_features])
y_pred_B_binary = (y_pred_B >= 0.5).astype(int)

# Performans karsilastirmasi
acc_A = accuracy_score(y_test, y_pred_A_binary)
acc_B = accuracy_score(y_test, y_pred_B_binary)

print('=' * 50)
print('        MODEL KARSILASTIRMASI')
print('=' * 50)
print(f'  Model A (korunan dahil):  Accuracy = {acc_A:.1%}')
print(f'  Model B (korunan haric):  Accuracy = {acc_B:.1%}')
print(f'  Fairness maliyeti:        {abs(acc_A - acc_B):.1%} fark')
print('=' * 50)

print(f'\nYorum:')
if abs(acc_A - acc_B) < 0.02:
    print('  Korunan ozellikleri cikarmak performansi neredeyse etkilemedi.')
    print('  -> Bu ozellikler zaten gereksiz - bias riski olmadan cikarilabilir.')
else:
    print(f'  Korunan ozellikleri cikarmak {abs(acc_A - acc_B):.1%} performans kaybina yol acti.')
    print('  -> Governance karari: Bu kayip kabul edilebilir mi?')
    print('  -> EU AI Act yuksek riskli sistemlerde fairness\'i zorunlu kiliyor.')

---
## Bolum 5: Disparate Impact Analizi

**Disparate Impact (Orantisiz Etki):** Bir karar surecinin, korunan bir gruba
orantisiz sekilde zarar vermesi.

**80% Kurali (Four-Fifths Rule):** ABD EEOC tarafindan kullanilan basit test.
Dezavantajli grubun secilme orani, avantajli grubun oraninin %80'inden azsa,
disparate impact var demektir.

Bu oran EU AI Act'ta dogrudan referans edilmese de, Article 10(2)(f) kapsaminda
"bias acisindan veri incelemesi" gereksinimi bu tur analizleri kapsar.

In [ ]:
# Disparate Impact analizi

def disparate_impact_ratio(y_pred, protected_attr, privileged_value, unprivileged_value):
    """Disparate Impact oranini hesapla.

    1.0 = Tam esitlik
    < 0.8 = Disparate impact var (80% kurali)
    > 1.0 = Dezavantajli grup lehine
    """
    privileged_mask = protected_attr == privileged_value
    unprivileged_mask = protected_attr == unprivileged_value

    rate_privileged = y_pred[privileged_mask].mean()
    rate_unprivileged = y_pred[unprivileged_mask].mean()

    return rate_unprivileged / rate_privileged if rate_privileged > 0 else np.nan

# Test seti uzerinde disparate impact
test_gender = df.loc[X_test.index, 'is_female']
test_age = df.loc[X_test.index, 'is_young']

# Cinsiyet bazinda
di_gender_A = disparate_impact_ratio(y_pred_A_binary, test_gender, 0, 1)
di_gender_B = disparate_impact_ratio(y_pred_B_binary, test_gender, 0, 1)

# Yas bazinda
di_age_A = disparate_impact_ratio(y_pred_A_binary, test_age, 0, 1)
di_age_B = disparate_impact_ratio(y_pred_B_binary, test_age, 0, 1)

print('=' * 58)
print('           DISPARATE IMPACT ANALIZI')
print('   (1.0 = esit, < 0.8 = orantisiz etki var)')
print('=' * 58)
print(f'                    Model A       Model B (fair)')
print(f'  Cinsiyet (K/E):   {di_gender_A:.3f}          {di_gender_B:.3f}')
print(f'  Yas (genc/diger): {di_age_A:.3f}          {di_age_B:.3f}')
print('=' * 58)

print('\nDegerlendirme:')
for name, di_a, di_b in [('Cinsiyet', di_gender_A, di_gender_B), ('Yas', di_age_A, di_age_B)]:
    status_a = '!! DI Ihlali' if di_a < 0.8 else 'Uygun'
    status_b = '!! DI Ihlali' if di_b < 0.8 else 'Uygun'
    print(f'  {name}: Model A -> {status_a} ({di_a:.3f}) | Model B -> {status_b} ({di_b:.3f})')

---
## Bolum 6: Gorsellestirme - Bias Raporu

Son olarak, bulgulari bir governance raporuna donusturelim.
Bu gorsellestirme, teknik olmayan paydaslara (yonetim kurulu,
uyum birimi) sunulabilecek formatta.

In [ ]:
# Bias Audit Ozet Raporu - Gorsellestirme

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('BIAS AUDIT RAPORU - German Credit Dataset\n'
             'EU AI Act Article 10 Uyum Degerlendirmesi',
             fontsize=15, fontweight='bold', y=1.02)

# 1. Katsayi karsilastirma
protected_coefs = coef_df[coef_df['Ozellik'].isin(['is_female', 'is_young', 'age'])]
axes[0,0].barh(protected_coefs['Ozellik'], protected_coefs['Katsayi'],
               color='#B2986C', edgecolor='#1C1E23')
axes[0,0].axvline(x=0, color='#1C1E23', linewidth=1)
axes[0,0].set_title('Korunan Ozelliklerin Katsayilari')
axes[0,0].set_xlabel('Standartlastirilmis Katsayi')

# 2. Disparate Impact gostergesi
di_data = pd.DataFrame({
    'Grup': ['Cinsiyet\n(Model A)', 'Cinsiyet\n(Model B)',
             'Yas\n(Model A)', 'Yas\n(Model B)'],
    'DI Orani': [di_gender_A, di_gender_B, di_age_A, di_age_B]
})
bar_colors = ['#B2986C' if v < 0.8 else '#686662' for v in di_data['DI Orani']]
axes[0,1].bar(di_data['Grup'], di_data['DI Orani'], color=bar_colors, edgecolor='#1C1E23')
axes[0,1].axhline(y=0.8, color='#1C1E23', linestyle='--', linewidth=1.5, label='80% esigi')
axes[0,1].set_title('Disparate Impact Oranlari')
axes[0,1].set_ylabel('DI Orani')
axes[0,1].legend()
axes[0,1].set_ylim(0, 1.3)

# 3. Model performans karsilastirma
models = ['Model A\n(bias dahil)', 'Model B\n(fair)']
accuracies = [acc_A, acc_B]
axes[1,0].bar(models, accuracies, color=['#686662', '#1C1E23'], edgecolor='#1C1E23')
axes[1,0].set_title('Model Performans Karsilastirmasi')
axes[1,0].set_ylabel('Accuracy')
axes[1,0].set_ylim(0, 1)
for i, v in enumerate(accuracies):
    axes[1,0].text(i, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# 4. Onay orani farklari (cinsiyet)
pred_A_df = pd.DataFrame({'pred': y_pred_A_binary, 'gender': test_gender.values})
pred_B_df = pd.DataFrame({'pred': y_pred_B_binary, 'gender': test_gender.values})

x_pos = np.arange(2)
width = 0.35
rates_A = pred_A_df.groupby('gender')['pred'].mean()
rates_B = pred_B_df.groupby('gender')['pred'].mean()

axes[1,1].bar(x_pos - width/2, [rates_A.get(0, 0), rates_A.get(1, 0)],
              width, label='Model A', color='#686662', edgecolor='#1C1E23')
axes[1,1].bar(x_pos + width/2, [rates_B.get(0, 0), rates_B.get(1, 0)],
              width, label='Model B (fair)', color='#1C1E23', edgecolor='#1C1E23')
axes[1,1].set_xticks(x_pos)
axes[1,1].set_xticklabels(['Erkek', 'Kadin'])
axes[1,1].set_title('Cinsiyete Gore Onay Oranlari')
axes[1,1].set_ylabel('Onay Orani')
axes[1,1].legend()
axes[1,1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## Bolum 7: Sonuc ve EU AI Act Baglantisi

### Bu Analizden Cikan Governance Aksiyonlari

**1. Article 10(2)(f) - Veri Setinde Bias Incelemesi**
Veri setinde cinsiyet ve yas bazinda farkli kredi onay oranlari tespit edildi.
Bu, egitim verisinin onyargi tasiyabilecegini gosterir.

**2. Article 10(2)(g) - Bias Azaltma Tedbirleri**
Model B (korunan ozellikler haric) daha adil sonuclar uretirken
performans kaybi sinirli kaldi. Bu, uygulanabilir bir bias azaltma stratejisi.

**3. Article 13 - Seffaflik ve Aciklanabilirlik**
Lineer regresyon katsayilari, modelin karar mekanizmasini dogrudan
aciklar. Bu, yuksek riskli sistemlerde gereken seffaflik seviyesini karsilar.

**4. ORIENT Framework - Evaluate Asamasi**
Bu notebook, ORIENT framework'unun Evaluate asamasina tekabul eder:
"Mevcut sistemde uyum aciklari nerede?" sorusuna veri odakli cevap.

### Sonraki Adimlar
- Navigate: Tespit edilen bias icin aksiyon plani olustur
- Track: Model fairness metriklerini duzenli olarak izle
- Daha gelismis fairness yontemleri: Equalized Odds, Calibration
- Streamlit ile interaktif Bias Audit araci gelistir

---
*Bu analiz Hexis AI Governance (hexis.center) tarafindan
EU AI Act uyum degerlendirmesi amaciyla hazirlanmistir.*